# 08 Report Analytics

## Days 11-13: Report-Level Behavioural Analytics

This notebook adds a simple, explainable analytics layer on top of the processed Power BI usage data. The goal is to compute report-level behavioural features, group reports into business-friendly segments, and apply diagnostic rules that help identify health risks.

The reusable implementation lives in `src/analytics/`. This notebook is the portfolio-friendly walkthrough layer.

## 1. Project Context

The forecasting pipeline estimates future report usage. This analytics layer is separate: it explains current report behaviour using processed semantic model tables, without changing forecasting logic or regenerating synthetic source data.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.analytics.report_features import build_report_features
from src.analytics.report_segmentation import build_report_segments
from src.analytics.report_diagnostics import build_report_diagnostics

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
SEGMENTS_DIR = PROJECT_ROOT / "outputs" / "segments"
DIAGNOSTICS_DIR = PROJECT_ROOT / "outputs" / "diagnostics"

for output_dir in [METRICS_DIR, SEGMENTS_DIR, DIAGNOSTICS_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Processed Input Tables

The workflow uses the best available processed tables. `mart_forecast_features.csv` is the richest daily usage mart, while the report views fact table supplies user-level repeat and concentration measures.

In [ ]:
daily_usage = pd.read_csv(PROCESSED_DIR / "mart_forecast_features.csv")
fact_report_views = pd.read_csv(PROCESSED_DIR / "fact_report_views.csv")
report_performance = pd.read_csv(PROCESSED_DIR / "mart_report_performance.csv")
dim_report = pd.read_csv(PROCESSED_DIR / "dim_report.csv")
dim_date = pd.read_csv(PROCESSED_DIR / "dim_date.csv")

input_tables = {
    "daily_usage": daily_usage.shape,
    "fact_report_views": fact_report_views.shape,
    "report_performance": report_performance.shape,
    "dim_report": dim_report.shape,
    "dim_date": dim_date.shape,
}
input_tables

## 3. Compute Report Features

The feature table contains one row per report. Core features include average daily views, repeat rate, top-user concentration, active days, performance summaries, and recent usage change.

In [ ]:
report_features = build_report_features(
    daily_adoption=daily_usage,
    fact_report_views=fact_report_views,
    report_performance=report_performance,
    dim_report=dim_report,
    dim_date=dim_date,
)

report_features_path = METRICS_DIR / "report_features.csv"
report_features.to_csv(report_features_path, index=False)

report_features.head()

## 4. Review `report_features.csv`

This quick profile checks the generated feature file and gives a compact view of the behavioural metrics.

In [ ]:
pd.read_csv(report_features_path).describe(include="all")

## 5. Create Report Segments

Reports are assigned to simple rule-based segments: `high_value`, `niche`, `at_risk`, or `inactive`. Quantiles are used where fixed business thresholds are not available.

In [ ]:
report_segments = build_report_segments(report_features)

report_segments_path = SEGMENTS_DIR / "report_segments.csv"
report_segments.to_csv(report_segments_path, index=False)

report_segments.head()

## 6. Review `report_segments.csv`

The segment distribution shows how many reports fall into each business-friendly group.

In [ ]:
pd.read_csv(report_segments_path)["report_segment"].value_counts()

## 7. Apply Diagnostic Rules

Diagnostics translate the features and segments into health flags for performance, engagement, dependency, and inactive risk.

In [ ]:
report_diagnostics = build_report_diagnostics(report_features, report_segments)

report_diagnostics_path = DIAGNOSTICS_DIR / "report_diagnostics.csv"
report_diagnostics.to_csv(report_diagnostics_path, index=False)

report_diagnostics.head()

## 8. Review `report_diagnostics.csv`

The main diagnostic gives one prioritized health status per report, with a short explanation for business users.

In [ ]:
pd.read_csv(report_diagnostics_path)["main_diagnostic"].value_counts()

## 9. Save Outputs

The three CSV outputs are saved to the expected project folders.

In [ ]:
for path in [report_features_path, report_segments_path, report_diagnostics_path]:
    print(path.relative_to(PROJECT_ROOT))

## 10. Brief Next Steps

Useful next steps would be to connect these outputs to the forecasting results, add stakeholder-approved thresholds, and later compare rule-based segments against clustering once the explainable version is accepted.